# FinRisk — XGBoost Market-Calibrated Impact Model Training

This notebook trains the XGBoost impact model for the FinRisk engine.

**Pipeline:**
1. Install dependencies
2. Fetch GDELT GKG bulk news data (1 year)
3. Fetch yfinance price data for 20 tickers + SPY
4. Compute abnormal returns (no look-ahead bias)
5. Map to 1–10 impact labels via percentile bins
6. Train XGBoost regressor with time-split
7. Evaluate vs rule-based baseline (MAE, RMSE, Spearman ρ)
8. Download `impact_xgb.pkl` + `impact_xgb_features.json`

**Runtime:** ~20–30 minutes on Colab free tier (CPU is fine, GPU not needed)

**After running:** Download the two model files and place them in `finrisk/data/models/`, then set `MODEL_IMPACT=xgboost` in your `.env`.

## Cell 1 — Install dependencies

In [ ]:
!pip install -q xgboost==2.0.3 yfinance==0.2.40 scipy==1.13.0 tqdm==4.66.2 \
               pyarrow==14.0.2 pandas==2.2.2 numpy==1.26.4 pyyaml==6.0.1 scikit-learn==1.5.0

import importlib, sys
for pkg in ['xgboost', 'yfinance', 'scipy', 'tqdm', 'pyarrow', 'yaml', 'sklearn']:
    try:
        importlib.import_module(pkg)
        print(f'  ✓ {pkg}')
    except ImportError as e:
        print(f'  ✗ {pkg}: {e}')
print('Dependencies ready.')

## Cell 2 — Configuration

In [ ]:
import os
from pathlib import Path

# ── Directories ───────────────────────────────────────────────────────────────
DATA_DIR    = Path('/content/finrisk_training')
MODELS_DIR  = DATA_DIR / 'models'
GDELT_OUT   = DATA_DIR / 'gdelt_gkg_filtered.parquet'
LABELED_OUT = DATA_DIR / 'labeled_events.parquet'
PERCNT_OUT  = DATA_DIR / 'impact_percentiles.json'
MODEL_OUT   = MODELS_DIR / 'impact_xgb.pkl'
FEATURES_OUT = MODELS_DIR / 'impact_xgb_features.json'

for d in [DATA_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Universe ──────────────────────────────────────────────────────────────────
TICKERS = [
    'AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'META', 'TSLA',
    'JPM',  'BAC',  'GS',   'V',     'JNJ',  'UNH',  'PFE',
    'XOM',  'CVX',  'CAT',  'BA',    'WMT',  'HD',
    'SPY',  # benchmark
]

UNIVERSE_TICKERS = [t for t in TICKERS if t != 'SPY']

# ── Company name fragments for GDELT filtering ────────────────────────────────
COMPANY_FILTERS = [
    'apple', 'microsoft', 'nvidia', 'alphabet', 'google', 'amazon',
    'meta', 'facebook', 'tesla', 'jpmorgan', 'jp morgan', 'bank of america',
    'goldman sachs', 'visa', 'johnson', 'unitedhealth', 'pfizer',
    'exxon', 'chevron', 'caterpillar', 'boeing', 'walmart', 'home depot',
]

# ── Model constants ───────────────────────────────────────────────────────────
EVENT_TYPES = [
    'Earnings', 'M&A', 'Regulatory', 'Macro', 'Geopolitical',
    'Credit', 'Commodity', 'Cyber', 'Leadership', 'Product',
]

SECTORS = [
    'Technology', 'Communication Services', 'Consumer Discretionary',
    'Financials', 'Health Care', 'Energy', 'Industrials', 'Consumer Staples',
]

TICKER_SECTOR = {
    'AAPL':'Technology',  'MSFT':'Technology',  'NVDA':'Technology',
    'GOOGL':'Communication Services', 'META':'Communication Services',
    'AMZN':'Consumer Discretionary',  'TSLA':'Consumer Discretionary', 'HD':'Consumer Discretionary',
    'JPM':'Financials',  'BAC':'Financials',  'GS':'Financials',  'V':'Financials',
    'JNJ':'Health Care', 'UNH':'Health Care', 'PFE':'Health Care',
    'XOM':'Energy',      'CVX':'Energy',
    'CAT':'Industrials', 'BA':'Industrials',
    'WMT':'Consumer Staples',
}

BASE_IMPACT = {
    'Earnings':5.0, 'M&A':5.5, 'Regulatory':6.0, 'Macro':6.0,
    'Geopolitical':7.0, 'Credit':8.0, 'Commodity':5.5,
    'Cyber':6.5, 'Leadership':4.5, 'Product':4.0,
}

LOOKBACK_DAYS = 365  # 1 year — enough data, reasonable download time

print(f'Configuration ready.')
print(f'  Data dir:   {DATA_DIR}')
print(f'  Tickers:    {len(TICKERS)} ({len(UNIVERSE_TICKERS)} universe + SPY)')
print(f'  Lookback:   {LOOKBACK_DAYS} days')

## Cell 3 — Fetch GDELT GKG data

Downloads ~35,000 GDELT 15-minute bulk files and filters for our 20 companies.  
**Expected time: 10–20 minutes** depending on Colab's network speed.

In [ ]:
import gzip, io, json, logging, time, re
from datetime import datetime, timedelta, timezone

import pandas as pd
import requests
from tqdm.notebook import tqdm

logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(levelname)s  %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('gdelt')

GDELT_BASE = 'http://data.gdeltproject.org/gdeltv2'
GDELT_MASTER = 'http://data.gdeltproject.org/gdeltv2/masterfilelist.txt'
MAX_RETRIES = 3
RETRY_WAIT  = 3


def _retry_get(url, timeout=45):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = requests.get(url, timeout=timeout)
            if r.status_code == 200:
                return r
            if r.status_code == 404:
                return None
            log.debug('HTTP %d for %s (attempt %d)', r.status_code, url, attempt)
        except requests.RequestException as e:
            log.debug('Request error %s: %s (attempt %d)', url, e, attempt)
        if attempt < MAX_RETRIES:
            time.sleep(RETRY_WAIT * attempt)
    return None


def _row_matches(themes, source):
    text = (themes + ' ' + source).lower()
    return any(f in text for f in COMPANY_FILTERS)


def _parse_gkg(raw_bytes):
    rows = []
    try:
        text = raw_bytes.decode('utf-8', errors='replace')
        for line in text.splitlines():
            if not line.strip():
                continue
            f = line.split('\t')
            if len(f) <= 15:
                continue
            themes = f[7] if len(f) > 7 else ''
            source = f[3] if len(f) > 3 else ''
            if not _row_matches(themes, source):
                continue
            rows.append({
                'GKGRECORDID': f[0],
                'DATE': f[1],
                'SourceCommonName': f[3] if len(f) > 3 else '',
                'DocumentIdentifier': f[4] if len(f) > 4 else '',
                'V2Themes': f[7] if len(f) > 7 else '',
                'V2Tone': f[15] if len(f) > 15 else '',
            })
    except Exception as e:
        log.debug('Parse error: %s', e)
    return rows


def _parse_gdelt_ts(s):
    try:
        return pd.Timestamp(datetime.strptime(str(s), '%Y%m%d%H%M%S').replace(tzinfo=timezone.utc))
    except:
        return pd.NaT


# ── Get file list ─────────────────────────────────────────────────────────────
end_dt   = datetime.now(tz=timezone.utc)
start_dt = end_dt - timedelta(days=LOOKBACK_DAYS)

log.info('Fetching GDELT master file list ...')
resp = _retry_get(GDELT_MASTER, timeout=120)
urls = []
if resp:
    for line in resp.text.splitlines():
        parts = line.split()
        if len(parts) >= 3 and parts[2].endswith('.gkg.csv.zip'):
            url = parts[2].strip()
            fname = url.split('/')[-1].replace('.gkg.csv.zip', '')
            try:
                fdt = datetime.strptime(fname, '%Y%m%d%H%M%S').replace(tzinfo=timezone.utc)
                if start_dt <= fdt <= end_dt:
                    urls.append(url)
            except ValueError:
                pass
    log.info('Master list: %d files in range', len(urls))
else:
    # Fallback: generate URLs from timestamps every 15 min
    log.warning('Master list unavailable — constructing URLs from timestamps')
    cur = start_dt
    while cur <= end_dt:
        minute = (cur.minute // 15) * 15
        snapped = cur.replace(minute=minute, second=0, microsecond=0)
        urls.append(f"{GDELT_BASE}/{snapped.strftime('%Y%m%d%H%M%S')}.gkg.csv.zip")
        cur += timedelta(minutes=15)
    log.info('Fallback: generated %d URLs', len(urls))

# ── Download and filter ───────────────────────────────────────────────────────
all_rows = []
skipped  = 0
# Progress bar for Colab
for url in tqdm(sorted(urls), desc='GDELT files', unit='file'):
    r = _retry_get(url, timeout=60)
    if r is None:
        skipped += 1
        continue
    try:
        with gzip.GzipFile(fileobj=io.BytesIO(r.content)) as gz:
            csv_bytes = gz.read()
        all_rows.extend(_parse_gkg(csv_bytes))
    except Exception as e:
        skipped += 1

log.info('Done: %d matching rows, %d files skipped', len(all_rows), skipped)

if not all_rows:
    raise RuntimeError('No GDELT rows fetched — check network and try again.')

gdelt_df = pd.DataFrame(all_rows)
gdelt_df['article_ts'] = gdelt_df['DATE'].apply(_parse_gdelt_ts)
gdelt_df = gdelt_df.dropna(subset=['article_ts']).sort_values('article_ts').reset_index(drop=True)
gdelt_df.to_parquet(GDELT_OUT, index=False)

print(f'\n✓ GDELT data saved: {len(gdelt_df):,} rows → {GDELT_OUT}')
print(f'  Date range: {gdelt_df["article_ts"].min().date()} → {gdelt_df["article_ts"].max().date()}')
gdelt_df.head(3)

## Cell 4 — Fetch yfinance price data

Downloads daily + 1-hour OHLCV for all 20 tickers + SPY.  
**Expected time: 2–5 minutes**

In [ ]:
import yfinance as yf
from tqdm.notebook import tqdm

end_str       = datetime.now(tz=timezone.utc).strftime('%Y-%m-%d')
daily_start   = (datetime.now(tz=timezone.utc) - timedelta(days=LOOKBACK_DAYS)).strftime('%Y-%m-%d')
hourly_start  = (datetime.now(tz=timezone.utc) - timedelta(days=59)).strftime('%Y-%m-%d')  # yfinance 60d limit

fetched_tickers = []

for ticker in tqdm(TICKERS, desc='Downloading prices', unit='ticker'):
    t = yf.Ticker(ticker)
    ok_daily = False
    ok_hourly = False

    # Daily
    for attempt in range(1, 4):
        try:
            df_d = t.history(start=daily_start, interval='1d', auto_adjust=True)
            if not df_d.empty:
                df_d.index = df_d.index.tz_convert('UTC') if df_d.index.tz else df_d.index.tz_localize('UTC')
                df_d.index.name = 'datetime'
                df_d.to_parquet(DATA_DIR / f'prices_daily_{ticker}.parquet')
                ok_daily = True
            break
        except Exception as e:
            if attempt < 3:
                time.sleep(2)

    # Hourly
    for attempt in range(1, 4):
        try:
            df_h = t.history(start=hourly_start, interval='1h', auto_adjust=True)
            if not df_h.empty:
                df_h.index = df_h.index.tz_convert('UTC') if df_h.index.tz else df_h.index.tz_localize('UTC')
                df_h.index.name = 'datetime'
                df_h.to_parquet(DATA_DIR / f'prices_1h_{ticker}.parquet')
                ok_hourly = True
            break
        except Exception as e:
            if attempt < 3:
                time.sleep(2)

    if ok_daily or ok_hourly:
        fetched_tickers.append(ticker)
    time.sleep(0.4)  # polite delay

print(f'\n✓ Prices downloaded for {len(fetched_tickers)}/{len(TICKERS)} tickers')
missing = [t for t in TICKERS if t not in fetched_tickers]
if missing:
    print(f'  Missing: {missing}')

# Quick sanity check: show SPY
spy = pd.read_parquet(DATA_DIR / 'prices_daily_SPY.parquet')
print(f'\n  SPY daily: {len(spy)} bars, {spy.index[0].date()} → {spy.index[-1].date()}')

## Cell 5 — Compute abnormal returns + build impact labels

For each GDELT news event at timestamp `t` for ticker `X`:  
- Entry price = first bar **strictly after** `t` (no look-ahead)  
- AR_1h = stock return over 1h − SPY return over 1h  
- AR_1d = stock return over 1d − SPY return over 1d  
- `impact_label` = percentile rank of `max(|AR_1h|, |AR_1d|)` → scaled to 1–10  

**Expected time: 5–10 minutes**

In [ ]:
import numpy as np
from tqdm.notebook import tqdm

# ── Ticker alias map for GDELT matching ───────────────────────────────────────
TICKER_ALIASES = {
    'AAPL':  ['apple', 'apple inc'],
    'MSFT':  ['microsoft', 'microsoft corp'],
    'NVDA':  ['nvidia', 'nvidia corp'],
    'GOOGL': ['alphabet', 'google', 'alphabet inc'],
    'AMZN':  ['amazon', 'amazon.com'],
    'META':  ['meta', 'facebook', 'meta platforms'],
    'TSLA':  ['tesla', 'tesla inc', 'tesla motors'],
    'JPM':   ['jpmorgan', 'jp morgan', 'jpmorgan chase'],
    'BAC':   ['bank of america', 'bofa', 'bankofamerica'],
    'GS':    ['goldman sachs', 'goldman'],
    'V':     ['visa', 'visa inc'],
    'JNJ':   ['johnson', 'johnson & johnson'],
    'UNH':   ['unitedhealth', 'united health'],
    'PFE':   ['pfizer'],
    'XOM':   ['exxon', 'exxonmobil', 'exxon mobil'],
    'CVX':   ['chevron'],
    'CAT':   ['caterpillar'],
    'BA':    ['boeing'],
    'WMT':   ['walmart', 'wal-mart'],
    'HD':    ['home depot'],
}


def _identify_tickers(v2themes, source):
    text = (v2themes + ' ' + source).lower()
    found = []
    for ticker in UNIVERSE_TICKERS:
        for alias in TICKER_ALIASES.get(ticker, [ticker.lower()]):
            if alias and alias in text:
                found.append(ticker)
                break
    return found


def _load_prices(ticker, interval='1h'):
    path = DATA_DIR / f'prices_{interval}_{ticker}.parquet'
    if not path.exists():
        return None
    df = pd.read_parquet(path)
    if df.index.tz is None:
        df.index = df.index.tz_localize('UTC')
    else:
        df.index = df.index.tz_convert('UTC')
    return df['Close'].sort_index()


def _next_price_after(prices, ts):
    future = prices[prices.index > ts]
    if future.empty:
        return None
    return float(future.iloc[0]), future.index[0]


def _price_in_window(prices, ts, td):
    window = prices[(prices.index > ts) & (prices.index <= ts + td)]
    if window.empty:
        return None
    return float(window.iloc[-1])


def _compute_ar(article_ts, ticker, h_prices, d_prices, spy_h, spy_d):
    """Compute 1h and 1d abnormal return. Returns dict or None."""
    sh = h_prices.get(ticker)
    sd = d_prices.get(ticker)
    if sh is None and sd is None:
        return None

    # Entry: first price strictly after article_ts
    entry = None
    if sh is not None:
        entry = _next_price_after(sh, article_ts)
    if entry is None and sd is not None:
        entry = _next_price_after(sd, article_ts)
    if entry is None:
        return None
    entry_price, entry_ts = entry

    # === LOOK-AHEAD CHECK ===
    assert entry_ts > article_ts, f'Look-ahead: entry_ts={entry_ts} <= article_ts={article_ts}'

    ar_1h = None
    if sh is not None:
        ep1h = _price_in_window(sh, article_ts, pd.Timedelta(hours=1))
        sp_s = _next_price_after(spy_h, article_ts) if spy_h is not None and len(spy_h) else None
        sp_e = _price_in_window(spy_h, article_ts, pd.Timedelta(hours=1)) if spy_h is not None else None
        if ep1h and sp_s and sp_e:
            sp_sp, sp_ts = sp_s
            assert sp_ts > article_ts, f'SPY look-ahead 1h'
            sr = (ep1h - entry_price) / entry_price if entry_price else 0.0
            mr = (sp_e - sp_sp) / sp_sp if sp_sp else 0.0
            ar_1h = sr - mr

    ar_1d = None
    pd_s = sd if sd is not None else sh
    spy_pd = spy_d if spy_d is not None else spy_h
    if pd_s is not None:
        ep1d = _price_in_window(pd_s, article_ts, pd.Timedelta(days=1))
        sp_s_d = _next_price_after(spy_pd, article_ts) if spy_pd is not None and len(spy_pd) else None
        sp_e_d = _price_in_window(spy_pd, article_ts, pd.Timedelta(days=1)) if spy_pd is not None else None
        if ep1d and sp_s_d and sp_e_d:
            sp_sp_d, sp_ts_d = sp_s_d
            assert sp_ts_d > article_ts, f'SPY look-ahead 1d'
            sr_d = (ep1d - entry_price) / entry_price if entry_price else 0.0
            mr_d = (sp_e_d - sp_sp_d) / sp_sp_d if sp_sp_d else 0.0
            ar_1d = sr_d - mr_d

    if ar_1h is None and ar_1d is None:
        return None

    abs_ar_max = max(abs(ar_1h) if ar_1h else 0.0, abs(ar_1d) if ar_1d else 0.0)
    return {'entry_ts': entry_ts, 'entry_price': entry_price,
            'ar_1h': ar_1h, 'ar_1d': ar_1d, 'abs_ar_max': abs_ar_max}


# ── Load all prices into memory ───────────────────────────────────────────────
print('Loading price data ...')
hourly_prices = {t: _load_prices(t, '1h') for t in UNIVERSE_TICKERS + ['SPY']}
daily_prices  = {t: _load_prices(t, 'daily') for t in UNIVERSE_TICKERS + ['SPY']}
spy_h = hourly_prices.get('SPY')
spy_d = daily_prices.get('SPY')

print(f'  SPY hourly bars: {len(spy_h) if spy_h is not None else "MISSING"}')
print(f'  SPY daily bars:  {len(spy_d) if spy_d is not None else "MISSING"}')

if spy_h is None and spy_d is None:
    raise RuntimeError('SPY price data missing — cannot compute abnormal returns.')

# ── Process each GDELT event ──────────────────────────────────────────────────
print(f'\nComputing abnormal returns for {len(gdelt_df):,} GDELT rows ...')
records = []
look_ahead_errors = 0

for _, row in tqdm(gdelt_df.iterrows(), total=len(gdelt_df), desc='Events'):
    article_ts = row['article_ts']
    if pd.isna(article_ts):
        continue

    tickers_in_row = _identify_tickers(str(row.get('V2Themes','')), str(row.get('SourceCommonName','')))
    for ticker in tickers_in_row:
        try:
            ar = _compute_ar(article_ts, ticker, hourly_prices, daily_prices, spy_h, spy_d)
        except AssertionError as e:
            look_ahead_errors += 1
            continue
        if ar is None:
            continue

        # Parse V2Tone
        tone_parts = str(row.get('V2Tone', '0,0,0')).split(',')
        try:
            t_overall  = float(tone_parts[0]) if tone_parts else 0.0
            t_positive = float(tone_parts[1]) if len(tone_parts) > 1 else 0.0
            t_negative = float(tone_parts[2]) if len(tone_parts) > 2 else 0.0
        except (ValueError, IndexError):
            t_overall = t_positive = t_negative = 0.0

        records.append({
            'GKGRECORDID':    row.get('GKGRECORDID',''),
            'article_ts':     article_ts,
            'ticker':         ticker,
            'SourceCommonName': row.get('SourceCommonName',''),
            'DocumentIdentifier': row.get('DocumentIdentifier',''),
            'V2Themes':       row.get('V2Themes',''),
            'V2Tone_raw':     row.get('V2Tone',''),
            'tone_overall':   t_overall,
            'tone_positive':  t_positive,
            'tone_negative':  t_negative,
            'entry_ts':       ar['entry_ts'],
            'entry_price':    ar['entry_price'],
            'ar_1h':          ar['ar_1h'],
            'ar_1d':          ar['ar_1d'],
            'abs_ar_max':     ar['abs_ar_max'],
        })

if look_ahead_errors > 0:
    print(f'  WARNING: {look_ahead_errors} look-ahead violations were skipped (should be 0)')

if not records:
    raise RuntimeError('No labeled events produced. Price data may not overlap with GDELT time range.')

events_df = pd.DataFrame(records)
print(f'\nProduced {len(events_df):,} event-ticker pairs')

# ── Map |AR| to impact labels via empirical percentiles ───────────────────────
PERCENTILE_BINS = [
    (0.0,  10.0, (1, 2)),
    (10.0, 30.0, (3, 4)),
    (30.0, 60.0, (4, 6)),
    (60.0, 80.0, (6, 8)),
    (80.0, 100.0, (8, 10)),
]

abs_ar = events_df['abs_ar_max'].values.astype(float)
p995   = float(np.percentile(abs_ar, 99.5))
abs_ar_clipped = np.clip(abs_ar, 0.0, p995)

pct_thresholds = [float(np.percentile(abs_ar_clipped, p)) for p, _, _ in PERCENTILE_BINS[1:]]

def _map_impact(v):
    for i, (_, _, (lo, hi)) in enumerate(PERCENTILE_BINS):
        thresh = pct_thresholds[i] if i < len(pct_thresholds) else float('inf')
        if v < thresh:
            start = pct_thresholds[i-1] if i > 0 else 0.0
            bw    = thresh - start
            frac  = (v - start) / bw if bw > 0 else 0.5
            return float(lo) + frac * (hi - lo)
    lo, hi = PERCENTILE_BINS[-1][2]
    scale  = min((v - pct_thresholds[-1]) / (pct_thresholds[-1] + 1e-9), 1.0)
    return float(lo) + scale * (hi - lo)

events_df['impact_label'] = events_df['abs_ar_max'].apply(_map_impact).clip(1.0, 10.0).round(1)

# Save percentiles for inference
import json
PERCNT_OUT.write_text(json.dumps({'percentile_thresholds': pct_thresholds, 'p99_5_winsor': p995}, indent=2))

events_df = events_df.sort_values('article_ts').reset_index(drop=True)
events_df.to_parquet(LABELED_OUT, index=False)

print(f'\n✓ Labeled events saved: {len(events_df):,} rows → {LABELED_OUT}')
print(f'\nImpact label distribution:')
bins   = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10.1]
labels = ['1-2','2-3','3-4','4-5','5-6','6-7','7-8','8-9','9-10']
counts, _ = np.histogram(events_df['impact_label'].values, bins=bins)
for lbl, cnt in zip(labels, counts):
    bar = '█' * min(cnt // max(1, len(events_df) // 40), 40)
    print(f'  {lbl}  {cnt:6d}  {bar}')
print(f'\nPer-ticker counts:')
print(events_df['ticker'].value_counts().to_string())

## Cell 6 — Feature engineering + XGBoost training

Time-based 70/15/15 split (no random shuffling — strict chronological order).  
**Expected time: 2–5 minutes**

In [ ]:
import pickle
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

df = pd.read_parquet(LABELED_OUT)
df = df.sort_values('article_ts').reset_index(drop=True)
print(f'Loaded {len(df):,} labeled events')

# ── Event type from V2Themes ──────────────────────────────────────────────────
def _infer_event_type(v2themes):
    text = v2themes.lower()
    if any(k in text for k in ['earn','revenue','eps','quarterly','guidance','profit']): return 'Earnings'
    if any(k in text for k in ['merger','acquisition','takeover','buyout','divest']):    return 'M&A'
    if any(k in text for k in ['regulation','sec_','antitrust','penalty','fine','lawsuit']): return 'Regulatory'
    if any(k in text for k in ['federal_reserve','interest_rate','inflation','gdp','unemployment']): return 'Macro'
    if any(k in text for k in ['sanction','war','tariff','military','geopolit','conflict']): return 'Geopolitical'
    if any(k in text for k in ['credit','debt','downgrade','bankrupt','bond','default']): return 'Credit'
    if any(k in text for k in ['oil','gas','commodity','opec','energy_price','crude']):   return 'Commodity'
    if any(k in text for k in ['cyber','hack','breach','ransomware','malware']):          return 'Cyber'
    if any(k in text for k in ['ceo','cfo','resign','appoint','executive','departure']):  return 'Leadership'
    if any(k in text for k in ['product','launch','recall','patent','drug','approval']):  return 'Product'
    return 'Earnings'


def _is_after_hours(ts):
    try:
        t = pd.Timestamp(ts)
        t = t.tz_localize('UTC') if t.tz is None else t.tz_convert('UTC')
        mo = t.replace(hour=13, minute=30, second=0)
        mc = t.replace(hour=20, minute=0,  second=0)
        return t.weekday() >= 5 or not (mo <= t < mc)
    except:
        return False


# ── Build features ────────────────────────────────────────────────────────────
print('Engineering features ...')
out = pd.DataFrame(index=df.index)

out['sentiment']     = (df['tone_overall'] / 10.0).clip(-1.0, 1.0)
out['abs_sentiment'] = out['sentiment'].abs()
out['tone_positive'] = df['tone_positive'].fillna(0.0)
out['tone_negative'] = df['tone_negative'].fillna(0.0).abs()
out['source_count']  = 1.0
out['is_after_hours'] = df['article_ts'].apply(lambda ts: float(_is_after_hours(ts)) if pd.notna(ts) else 0.0)

# Days since last event per ticker (no look-ahead)
df_s = df.sort_values(['ticker','article_ts']).copy()
df_s['prev_ts'] = df_s.groupby('ticker')['article_ts'].shift(1)
df_s['days_since_last'] = ((df_s['article_ts'] - df_s['prev_ts'])
                           .dt.total_seconds().div(86400)
                           .fillna(30.0).clip(0.0, 90.0))
out['days_since_last'] = df_s['days_since_last'].reindex(df.index).fillna(30.0)

# Event type one-hot
event_types_col = df['V2Themes'].apply(_infer_event_type)
for et in EVENT_TYPES:
    col = f"event_type_{et.lower().replace('&','and').replace(' ','_')}"
    out[col] = (event_types_col == et).astype(float)

# Sector one-hot
sectors_col = df['ticker'].map(TICKER_SECTOR).fillna('Technology')
for sec in SECTORS:
    col = f"sector_{sec.lower().replace(' ','_')}"
    out[col] = (sectors_col == sec).astype(float)

feature_columns = list(out.columns)
print(f'Features ({len(feature_columns)}): {feature_columns[:8]} ...')

X = out.values.astype(float)
y = df['impact_label'].values.astype(float)

# ── Time-based split — NO random shuffling ────────────────────────────────────
n = len(df)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

X_train, y_train = X[:train_end],       y[:train_end]
X_val,   y_val   = X[train_end:val_end], y[train_end:val_end]
X_test,  y_test  = X[val_end:],          y[val_end:]

print(f'\nSplit: train={len(X_train):,}  val={len(X_val):,}  test={len(X_test):,}')
print(f'  Train covers: up to {df.iloc[train_end]["article_ts"].date()}')
print(f'  Test  covers: {df.iloc[val_end]["article_ts"].date()} → {df["article_ts"].max().date()}')

# ── Train ─────────────────────────────────────────────────────────────────────
print('\nTraining XGBoost ...')
model = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='rmse',
    early_stopping_rounds=20,
    verbosity=0,
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
print(f'Best iteration: {model.best_iteration}')

# ── Evaluate ──────────────────────────────────────────────────────────────────
y_pred = np.clip(model.predict(X_test), 1.0, 10.0)

# Rule-based baseline
test_et = event_types_col.iloc[val_end:]
test_sent = out['sentiment'].iloc[val_end:].values
baseline = np.array([
    np.clip(BASE_IMPACT.get(et, 5.0) * (0.5 + 0.5*abs(s)) * 0.7 * 0.9, 1.0, 10.0)
    for et, s in zip(test_et, test_sent)
])

xgb_mae,  xgb_rmse  = mean_absolute_error(y_test, y_pred), float(np.sqrt(mean_squared_error(y_test, y_pred)))
base_mae, base_rmse = mean_absolute_error(y_test, baseline), float(np.sqrt(mean_squared_error(y_test, baseline)))
xgb_rho,  _  = spearmanr(y_test, y_pred)
base_rho, _  = spearmanr(y_test, baseline)

print(f'\n{"="*58}')
print(f'  {"Model":<30}  {"MAE":>6}  {"RMSE":>6}  {"Spearman ρ":>10}')
print(f'  {"-"*56}')
print(f'  {"Rule-based baseline":<30}  {base_mae:6.3f}  {base_rmse:6.3f}  {base_rho:+10.3f}')
print(f'  {"XGBoost":<30}  {xgb_mae:6.3f}  {xgb_rmse:6.3f}  {xgb_rho:+10.3f}')
print(f'{"="*58}')
improvement = (base_mae - xgb_mae) / base_mae * 100 if base_mae > 0 else 0
print(f'  MAE improvement over rule-based: {improvement:+.1f}%')

# Feature importances
print(f'\nTop 10 feature importances:')
feat_imp = sorted(zip(feature_columns, model.feature_importances_), key=lambda x: -x[1])
for feat, imp in feat_imp[:10]:
    bar = '█' * int(imp * 60)
    print(f'  {feat:<42} {imp:.4f}  {bar}')

# ── Save ──────────────────────────────────────────────────────────────────────
MODELS_DIR.mkdir(parents=True, exist_ok=True)

with open(MODEL_OUT, 'wb') as f:
    pickle.dump(model, f, protocol=pickle.HIGHEST_PROTOCOL)

meta = {
    'feature_columns': feature_columns,
    'event_types': EVENT_TYPES,
    'sectors': SECTORS,
    'n_train': int(len(X_train)),
    'n_val': int(len(X_val)),
    'n_test': int(len(X_test)),
    'best_iteration': int(model.best_iteration),
    'test_mae': float(xgb_mae),
    'test_rmse': float(xgb_rmse),
    'test_spearman_rho': float(xgb_rho),
    'baseline_mae': float(base_mae),
    'baseline_spearman_rho': float(base_rho),
}
FEATURES_OUT.write_text(json.dumps(meta, indent=2))

print(f'\n✓ Model saved:    {MODEL_OUT}')
print(f'✓ Features saved: {FEATURES_OUT}')

## Cell 7 — Download model files

Run this cell to download `impact_xgb.pkl` and `impact_xgb_features.json` to your local machine.

**After downloading:**
1. Put both files in `finrisk/data/models/`
2. Add `MODEL_IMPACT=xgboost` to `finrisk/.env`
3. Restart the API server — it will automatically load XGBoost

In [ ]:
from google.colab import files
import os

print('Downloading model artifacts ...')
files.download(str(MODEL_OUT))
files.download(str(FEATURES_OUT))
files.download(str(PERCNT_OUT))
print('Done! Place these 3 files in finrisk/data/models/')

# Also print evaluation summary for your presentation slide
print(f'\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'  COPY THIS FOR YOUR EVALUATION SLIDE')
print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
feat_meta = json.loads(FEATURES_OUT.read_text())
print(f'  Training samples:   {feat_meta["n_train"]:,}')
print(f'  Test samples:       {feat_meta["n_test"]:,}')
print(f'  XGBoost MAE:        {feat_meta["test_mae"]:.3f}')
print(f'  XGBoost RMSE:       {feat_meta["test_rmse"]:.3f}')
print(f'  XGBoost Spearman ρ: {feat_meta["test_spearman_rho"]:+.3f}')
print(f'  Baseline MAE:       {feat_meta["baseline_mae"]:.3f}')
print(f'  Baseline Spearman:  {feat_meta["baseline_spearman_rho"]:+.3f}')
print(f'  Best iteration:     {feat_meta["best_iteration"]}')
print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')

## Cell 8 — (Optional) Activate XGBoost on your local machine

Run these commands in a terminal **after** placing the downloaded files in `finrisk/data/models/`:

```bash
# 1. Activate the model
echo 'MODEL_IMPACT=xgboost' >> finrisk/.env

# 2. Verify it loads correctly
cd finrisk
.venv/bin/python -c "
from app.engine.xgboost_model import XGBoostImpactModel
m = XGBoostImpactModel()
ok = m.load()
print('XGBoost loaded:', ok)
if ok:
    features = m.get_feature_vector({}, sentiment=-0.7, event_type='Geopolitical')
    score = m.predict(features)
    print(f'Test prediction (Geopolitical, -0.7 sentiment): {score:.2f}/10')
"

# 3. Restart the API (Ctrl+C the running server, then)
.venv/bin/uvicorn app.main:app --host 0.0.0.0 --port 8000 --reload
```

You should see this in the API logs:
```
ImpactCalculator: using XGBoost model from data/models/impact_xgb.pkl
```